# Lesson 01 Lab — Operator Boundaries and the Cost of Small Kernels

**Puzzle:** When fusion, launch count, and memory traffic change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates fusion, launch count, and memory traffic and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

An eager expression may read and write a full tensor at every operator boundary. A Triton program can keep the intermediate value in registers and store once. The useful comparison therefore freezes the algebra and separates launch count, requested bytes, and latency.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["fusion, launch count, and memory traffic"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

A faster fused kernel does not prove every operator should be rewritten. Very small inputs, compiler fusion, or a strong library kernel can erase the advantage.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 1
LESSON_TITLE = 'Operator Boundaries and the Cost of Small Kernels'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260814
}


## 5. Freeze the experiment

**Experiment:** Compare one Triton affine kernel with the equivalent two-operation eager PyTorch CUDA path on the same tensor.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.02108799945563078,
  "secondary": 0.018880000337958336,
  "max_abs_error": 4.76837158203125e-07,
  "passed": true,
  "details": {
    "triton_samples_ms": [
      0.03097599931061268,
      0.02409599907696247,
      0.023391999304294586,
      0.022272000089287758,
      0.019840000197291374,
      0.021568000316619873,
      0.022495999932289124,
      0.022463999688625336,
      0.02067199908196926,
      0.020927999168634415,
      0.021247999742627144,
      0.020160000771284103,
      0.021536000072956085,
      0.02038400061428547,
      0.021344000473618507,
      0.020128000527620316,
      0.01881599985063076,
      0.019551999866962433,
      0.01929599978029728,
      0.02070399932563305
    ],
    "pytorch_samples_ms": [
      0.020447999238967896,
      0.0197759997099638,
      0.01942400075495243,
      0.01865600049495697,
      0.01881599985063076,
      0.018303999677300453,
      0.019519999623298645,
      0.01974399946630001,
      0.0185920000076

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Triton median | 0.0211 ms |
| PyTorch eager median | 0.0189 ms |
| Maximum absolute error | 4.768e-07 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The fused Triton affine kernel took 0.0211 ms versus 0.0189 ms for the two-operation eager path, with max error 4.77e-07.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Fuse only after measurements show that intermediate traffic or launch overhead matters for the production shape.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 1,
  "title": "Operator Boundaries and the Cost of Small Kernels",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260814
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.02108799945563078,
    "secondary": 0.018880000337958336,
    "max_abs_error": 4.76837158203125e-07,
    "passed": true,
    "details": {
      "triton_samples_ms": [
        0.03097599931061268,
        0.02409599907696247,
        0.023391999304294586,
        0.022272000089287758,
        0.019840000197291374,
        0.021568000316619873,
        0.022495999932289124,
        0.022463999688625336,
        0.02067199908196926,
        0.020927999168634415,
        0.021247999742627144,
        0.020160000771284103,
        0.021536000072956085,


## 10. Make the bounded decision

> Fuse only after measurements show that intermediate traffic or launch overhead matters for the production shape.

**Failure analysis:** A faster fused kernel does not prove every operator should be rewritten. Very small inputs, compiler fusion, or a strong library kernel can erase the advantage.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
